# 02 - Model Training
## BiLSTM + Contrastive Learning
**Team:** Gradient_Issues

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from scripts.preprocess import DataLoader
from models.encoder import TrafficEncoderBiLSTM, Trainer
from models.losses import CombinedLoss
from utils.helpers import plot_training_curves, plot_confusion_matrix

In [ ]:
dataloader = DataLoader('../data/5g_traffic_classification.csv')
train_ds = dataloader.get_tf_dataset(*dataloader.train_data, shuffle=True)
val_ds = dataloader.get_tf_dataset(*dataloader.val_data, shuffle=False)
test_ds = dataloader.get_tf_dataset(*dataloader.test_data, shuffle=False)

num_classes = len(dataloader.label_encoder.classes_)
label_names = dataloader.label_encoder.classes_
print(f"Classes: {list(label_names)}")

### Supervised BiLSTM

In [ ]:
model = TrafficEncoderBiLSTM(num_classes=num_classes, embedding_dim=32)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

trainer = Trainer(model, optimizer, loss_fn)
history = trainer.fit(train_ds, val_ds, epochs=100, patience=15)

plot_training_curves(history, 'BiLSTM Supervised', '../results/06_bilstm_training_curves.png')

In [ ]:
test_loss, test_acc = trainer.validate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

### Contrastive BiLSTM

In [ ]:
class ContrastiveTrainer(Trainer):
    def train_step(self, x, y):
        with tf.GradientTape() as tape:
            logits, embeddings = self.model(x, training=True)
            loss = self.loss_fn(y, (logits, embeddings))
        gradients = tape.gradient(loss, self.model.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.model.trainable_weights))
        return loss

    def validate(self, val_dataset):
        val_losses, val_accs = [], []
        for x_batch, y_batch in val_dataset:
            logits, embeddings = self.model(x_batch, training=False)
            loss = self.loss_fn(y_batch, (logits, embeddings))
            preds = tf.argmax(logits, axis=1)
            acc = tf.reduce_mean(tf.cast(tf.equal(preds, y_batch), tf.float32))
            val_losses.append(loss.numpy())
            val_accs.append(acc.numpy())
        return np.mean(val_losses), np.mean(val_accs)

model_con = TrafficEncoderBiLSTM(num_classes=num_classes, embedding_dim=32)
optimizer_con = tf.keras.optimizers.Adam(learning_rate=0.001)
combined_loss = CombinedLoss(ce_weight=1.0, con_weight=0.5, temperature=0.07)

trainer_con = ContrastiveTrainer(model_con, optimizer_con, combined_loss)
history_con = trainer_con.fit(train_ds, val_ds, epochs=100, patience=15)

plot_training_curves(history_con, 'BiLSTM Contrastive', '../results/08_contrastive_training_curves.png')

In [ ]:
test_loss_con, test_acc_con = trainer_con.validate(test_ds)
print(f"Contrastive Test Loss: {test_loss_con:.4f}")
print(f"Contrastive Test Accuracy: {test_acc_con:.4f}")